# Trabajo Fin de Máster  
### Análisis de la Ciudad mediante Aprendizaje Supervisado  
#### Detección Automática de Tipologías Residenciales y Patrones de Cerramiento: Interpretabilidad vs Rendimiento

**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**

> **Autor:** David Fernández Martínez    
> **Email personal:** david.fernxndez.martinez@gmail.com  
> **Email académico:** davidfm8@correo.ugr.es  
> **LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://www.linkedin.com/in/david-fern%C3%A1ndez-mart%C3%ADnez/)  
> **GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Metodología para el análisis de interpretabilidad

### 📝 Descripción del notebook
TO COMPLETE...

### Indice de contenidos
TO COMPLETE...

# Configuración de entorno e *imports*

Este proyecto ha sido realizado en un entorno Anaconda con la versión 3.11.15 de *Python*. Las versiones de las librerias requeridas se encuentran en el fichero *requirements.txt*.

En esta sección se importan las librerias necesarias para la ejecución de este fichero jupyter notebook, se activa el *reload* de módulos externos y se configuran aspectos globales y de reproducibilidad.

In [36]:
# jupyter extensions to automatically reload external modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
import warnings
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Configuration object
from src.config import cfg

# Production training method
from src.production_training import train_final_model

**Import troubleshooting**

If the `src` imports fail when running this notebook in a different environment,
uncomment and execute the following cell:

```python
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [51]:
# Global configuration
sns.set_theme(style="ticks", context="notebook")
plt.rcParams["font.family"] = "sans-serif"
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

In [52]:
# Reproducibility
SEED = cfg.SEED 
np.random.seed(SEED)
random.seed(SEED)

# 1. Análisis de interpretabilidad global

# 1.1 Entrenamiento de modelos sobre todo el conjunto de datos

En la fase de evaluación de rendimiento se ha empleado una estrategia de *Nested Cross Validation*  con el objetivo de estimar de forma realista la capacidad de generalización de los distintos algoritmos. Este enfoque permite evaluar cómo se comporta un determinado algoritmo al aprender de un conjunto de entrenamiento y generalizar ante datos no observados, simulando el escenario de producción, en el que el modelo se entrena con la información disponible hasta un determinado momento y realiza predicciones sobre los nuevos datos que ingresan al sistema.

El objetivo del análisis de interpretabilidad, sin embargo, es diferente. En este caso, no se pretende estimar la capacidad de generalización, sino comprender el comportamiento global aprendido por el modelo a partir de los datos. Por este motivo, la interpretabilidad se estudia a partir de modelos entrenados sobre todo el conjunto de datos disponible, maximizando la información utilizada para capturar la estructura subyacente del problema.
De esta forma se pretende explicar el modelo final que se desplegaría en producción, el cual se entrena con todos los datos históricos disponibles.

En la *Nested Cross Validation* se obtiene un modelo óptimo para cada partición externa, donde cada uno de ellos se configura con un conjunto de hiperparámetros optimizados específicamente para el subconjunto de entrenamiento correspondiente a dicha partición. Como se ha mencionado previamente, el análisis de interpretabilidad requiere disponer de un único modelo final. Una alternativa natural para la selección de hiperparámetros consistiría en utilizar la moda de los hiperparámetros obtenidos en las distintas particiones de la *Nested Cross Validation*. Sin embargo, este enfoque no garantiza la optimalidad global, ni necesariamente refleja la mejor configuración sobre el conjunto completo de datos.

Para obtener la configuración óptima sobre el conjunto completo de datos, se realiza una búsqueda de hiperparámetros utilizando el mismo espacio de búsqueda definido en la fase de evaluación de rendimiento. Esta estrategia corresponde al procedimiento estándar en el despliegue de modelos en producción, donde el objetivo es optimizar el rendimiento del modelo utilizando toda la información disponible hasta el momento.

Para llevar a cabo esta optimización, se aplica una validación cruzada estándar que utiliza exactamente las mismas particiones empleadas en el bucle externo (*Outer Loop*) de la *Nested Cross Validation*, garantizando así la reprodubilidad en la construcción de los modelos finales.

A continuación se entrenan todos los modelos mediante la funcion *train_final_model()* ubicada en el módulo *src/production_training.py*. Los modelos se almacenan en el directorio *output/models* en formato .pkl.

In [53]:
################################
# Multinomial Logistic Regression
################################

LR_model = LogisticRegression(
    class_weight="balanced",
    solver="lbfgs",
    penalty="l2",
    random_state = SEED
)


LR_param_grid = {
    "C": [10, 100, 1000, 10000],
}

LR_model = train_final_model(cfg, LR_model, LR_param_grid, "Logistic_Regression")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Logistic_Regression
CV Folds       : 5

Hyperparameter Grid:
C                        : [10, 100, 1000, 10000]
Fitting 5 folds for each of 4 candidates, totalling 20 fits

--------------------------------------------------------------------------------
Total time      : 0.08 seconds
--------------------------------------------------------------------------------
Best params for Logistic_Regression:
{'C': 10}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [54]:
################################
# Decision Tree
################################

DT_model = DecisionTreeClassifier(
    class_weight = "balanced",
    max_depth = 6,
    random_state = SEED
)

DT_param_grid = {
    "max_leaf_nodes": [10, 15, 20, 25, 30],
    "min_samples_leaf": [5, 10, 15],
    "criterion": ["gini", "entropy"],
}
DT_model = train_final_model(cfg, DT_model, DT_param_grid, "Decision_Tree")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Decision_Tree
CV Folds       : 5

Hyperparameter Grid:
max_leaf_nodes           : [10, 15, 20, 25, 30]
min_samples_leaf         : [5, 10, 15]
criterion                : ['gini', 'entropy']
Fitting 5 folds for each of 30 candidates, totalling 150 fits

--------------------------------------------------------------------------------
Total time      : 0.18 seconds
--------------------------------------------------------------------------------
Best params for Decision_Tree:
{'criterion': 'entropy', 'max_leaf_nodes': 15, 'min_samples_leaf': 5}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [55]:
################################
# SVM With RBF Kernel
################################

SVM_model = SVC(
    kernel = "rbf",
    decision_function_shape = 'ovr',
    class_weight = "balanced",
    random_state = SEED
)

SVM_param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.1]
}

SVM_model = train_final_model(cfg, SVM_model, SVM_param_grid, "SVM")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : SVM
CV Folds       : 5

Hyperparameter Grid:
C                        : [0.1, 1, 10, 100]
gamma                    : ['scale', 'auto', 0.01, 0.1]
Fitting 5 folds for each of 16 candidates, totalling 80 fits

--------------------------------------------------------------------------------
Total time      : 0.22 seconds
--------------------------------------------------------------------------------
Best params for SVM:
{'C': 10, 'gamma': 'auto'}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [56]:
################################
# Random Forest
################################

RF_model = RandomForestClassifier(
        class_weight = "balanced_subsample",
        random_state = SEED
)


RF_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, None],
    "max_features": ["sqrt", 0.3, 0.4],
    "criterion": ['gini', 'entropy'],
    "min_samples_split": [2, 5],
}  

RF_model = train_final_model(cfg, RF_model, RF_param_grid, "Random_Forest")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Random_Forest
CV Folds       : 5

Hyperparameter Grid:
n_estimators             : [100, 200, 300]
max_depth                : [5, 10, None]
max_features             : ['sqrt', 0.3, 0.4]
criterion                : ['gini', 'entropy']
min_samples_split        : [2, 5]
Fitting 5 folds for each of 108 candidates, totalling 540 fits

--------------------------------------------------------------------------------
Total time      : 19.58 seconds
--------------------------------------------------------------------------------
Best params for Random_Forest:
{'criterion': 'gini', 'max_depth': 10, 'max_features': 0.3, 'min_samples_split': 5, 'n_estimators': 100}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
-----------------------------------------------------------------------------

In [57]:
################################
# XGBoost
################################

XG_model = XGBClassifier(
        random_state = SEED,
        sampling_method = "uniform",
        objective= "multi:softmax",
        eval_metric="mlogloss",
)

XG_param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [5, 10],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 0.3],
    "reg_lambda": [1, 5]
}

XG_model = train_final_model(cfg, XG_model, XG_param_grid, "XGBoost", use_balanced_weights=True)


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : XGBoost
CV Folds       : 5

Hyperparameter Grid:
n_estimators             : [100, 300]
max_depth                : [5, 10]
learning_rate            : [0.01, 0.1]
subsample                : [0.8, 1.0]
colsample_bytree         : [0.8, 1.0]
gamma                    : [0, 0.3]
reg_lambda               : [1, 5]
Using balanced sample weights
Fitting 5 folds for each of 128 candidates, totalling 640 fits

--------------------------------------------------------------------------------
Total time      : 11.93 seconds
--------------------------------------------------------------------------------
Best params for XGBoost:
{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'reg_lambda': 1, 'subsample': 0.8}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJEC

Ejemplo de carga de un modelo y prediccion de muestras:

In [81]:
# LOAD MODEL

import joblib
saved_artifacts = joblib.load("../output/models/XGBoost_model.pkl")

model = saved_artifacts["model"]
encoder = saved_artifacts["label_encoder"]
model.classes_

array([0, 1, 2, 3, 4])

In [83]:
df = pd.read_csv("../data/processed/model_data.csv")

X = df.drop(columns=["CC", "URB"])
y = df["URB"]

n = 5 

for i in range(n):
    X_row = X.iloc[[i]]  
    y_true = y.iloc[i]

    y_pred_encoded = model.predict(X_row)[0]
    y_pred = encoder.inverse_transform([y_pred_encoded])[0]

    print(f"Fila {i}")
    print(f"  Real      : {y_true}")
    print(f"  Prediccion Encoded: {y_pred_encoded}")
    print(f"  Predicción Decoded: {y_pred}")
    print("-" * 40)


Fila 0
  Real      : 1
  Prediccion Encoded: 0
  Predicción Decoded: 1
----------------------------------------
Fila 1
  Real      : 4
  Prediccion Encoded: 3
  Predicción Decoded: 4
----------------------------------------
Fila 2
  Real      : 4
  Prediccion Encoded: 3
  Predicción Decoded: 4
----------------------------------------
Fila 3
  Real      : 4
  Prediccion Encoded: 3
  Predicción Decoded: 4
----------------------------------------
Fila 4
  Real      : 4
  Prediccion Encoded: 3
  Predicción Decoded: 4
----------------------------------------


# 2. Analisis de interpretabilidad local

TO COMPLETE...